In [2]:
import numpy as np
import pandas as pd
import torch
from torch import nn 

In [3]:
def train_test_split(X, y, test_size:float = None, train_size:float = None):
	if X.shape[0] != y.shape[0]:
		raise Exception("Size mismatch of X and y")
	else:
		total_rows=X.shape[0]
	if test_size == None and train_size == None:
		test_size=0.25
		train_size = 1 - test_size
	elif test_size == None:
		test_size = 1 - train_size
	elif train_size == None:
		train_size = 1 - test_size

	mark = int(total_rows*train_size)
	return X[:mark], y[:mark], X[mark:], y[mark:]


In [4]:
# from sklearn.model_selection import train_test_split
df = pd.read_csv('data.csv')
df.head()
X = df.drop(columns=['price']).to_numpy()
y = df['price'].to_numpy()

In [5]:
X_train, y_train, X_test, y_test = train_test_split(X, y, 0.2)
# Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train

array([[-1.4598444 , -1.78017249],
       [-1.20410524, -0.83074716],
       [-0.69262691, -0.83074716],
       [-0.18114858,  0.11867817],
       [ 0.33032976,  0.11867817],
       [ 0.67131531,  1.06810349],
       [ 1.01230086,  1.06810349],
       [ 1.52377919,  1.06810349]])

In [6]:
class Perceptron():
	def __init__(self, features_size):
		self.weights = torch.tensor(np.random.random(features_size), requires_grad=True, dtype=torch.float32)
		self.bias = torch.tensor(np.random.random(), requires_grad=True, dtype=torch.float32)

	def forward(self, features):
		features = torch.tensor(features, dtype=torch.float32)
		# self.model_pred = torch.empty(features.shape[0], dtype=torch.float32)
		self.model_pred = (features @ self.weights) + self.bias
		return self.model_pred
			
	def rmse(self, y_true):
		y_true = torch.tensor(y_true, dtype=torch.float32)
		if y_true.shape != self.model_pred.shape:
			raise Exception("Shape mismatch(y_true and y_pred are not same shape)")
		num = y_true.numel()
		self.loss = torch.mean((y_true-self.model_pred)**2).sqrt()
		return self.loss
	
	def backpropagation(self, lr=0.01):
		with torch.no_grad():
			# for i in range(len(self.weights)):
			# 	self.weights[i] = self.weights[i] - (lr * self.weights.grad[i])
			self.weights -= lr * self.weights.grad
			self.bias -= (lr * self.bias.grad)




In [7]:
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
	def __init__(self, features, labels):
		self.features = features
		self.labels = labels
	
	def __len__(self):
		return len(self.features)
	
	def __getitem__(self, idx):
		return self.features[idx], self.labels[idx]

In [9]:
train_set = CustomDataset(X_train, y_train)

In [10]:
train_loader = DataLoader(train_set, batch_size=1, shuffle=True)

In [25]:
model = Perceptron(X_train.shape[1])
epochs = 500
for i in range(epochs):
	losses=[]
	for batch_feature, batch_label in train_loader:
		y_pred = (model.forward(batch_feature))

		# Calculate loss

		loss = model.rmse(batch_label)
		losses.append(loss)
		# print("loss:", loss)

		model.weights.grad = None
		model.bias.grad=None
		# Calculating loss
		model.loss.backward()

		# update weights 
		model.backpropagation(lr=100)

	print(f"Avg loss in epoch {i + 1}: {sum(losses)/len(losses)}")
	losses.clear()

# print(model.bias.grad)
# print(model.bias)



C:\Users\LOQ\AppData\Local\Temp\ipykernel_7208\1663793951.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  features = torch.tensor(features, dtype=torch.float32)
C:\Users\LOQ\AppData\Local\Temp\ipykernel_7208\1663793951.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_true = torch.tensor(y_true, dtype=torch.float32)


Avg loss in epoch 1: 132874.84375
Avg loss in epoch 2: 132074.84375
Avg loss in epoch 3: 131274.84375
Avg loss in epoch 4: 130474.84375
Avg loss in epoch 5: 129674.84375
Avg loss in epoch 6: 128874.8359375
Avg loss in epoch 7: 128074.84375
Avg loss in epoch 8: 127274.8359375
Avg loss in epoch 9: 126474.84375
Avg loss in epoch 10: 125674.8359375
Avg loss in epoch 11: 124874.84375
Avg loss in epoch 12: 124074.84375
Avg loss in epoch 13: 123274.8359375
Avg loss in epoch 14: 122474.8359375
Avg loss in epoch 15: 121674.84375
Avg loss in epoch 16: 120874.8359375
Avg loss in epoch 17: 120074.84375
Avg loss in epoch 18: 119274.8359375
Avg loss in epoch 19: 118474.8359375
Avg loss in epoch 20: 117674.84375
Avg loss in epoch 21: 116874.8359375
Avg loss in epoch 22: 116074.8359375
Avg loss in epoch 23: 115274.84375
Avg loss in epoch 24: 114474.84375
Avg loss in epoch 25: 113674.8359375
Avg loss in epoch 26: 112874.84375
Avg loss in epoch 27: 112074.8359375
Avg loss in epoch 28: 111274.8359375
Avg

In [ ]:
with torch.no_grad():
	model.weights[0] -= 0.1*model.weights.grad[0]
	print(model.weights)



tensor([4.6396, 4.5645], requires_grad=True)


In [ ]:
with torch.no_grad():
	model.weights[0] = 2 - 0.01 * -4.7684e-07

model.weights[0]

tensor(2., grad_fn=<SelectBackward0>)

In [ ]:
y_t = np.array([1])
y_p = np.array([0.5])
((sum((y_t - y_p)**2))**0.5)/len(y_t)


np.float64(0.5)

In [ ]:
t = torch.empty(X_train.shape[0])
for i in range(5):
	t[i] += i
t

tensor([-14.7021,  -8.2422,  -4.9128,   2.7119,   6.0413,   7.8895,   9.4425,
         11.7720])